In [ ]:
# IMPORTS
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Directory containing index_record.txt and snapshots_cts* folders
RESULTS_DIR = os.path.join(os.getcwd())

def snapshot_path_continuous(run_index):
    base = os.path.join(RESULTS_DIR, f'snapshots_cts{run_index}')
    return os.path.join(base, f'snapshots_cts{run_index}_s1.h5')



In [ ]:
# BULK PROCESSING AND VISUALISATION
Ma_indices = []
Ma_list = []
Ma_c_max = 1e7

index_file = os.path.join(RESULTS_DIR, 'index_record.txt')
with open(index_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            index_str, param_str = line.split()
            if float(param_str) > Ma_c_max:
                continue
            Ma_indices.append(int(index_str))
            Ma_list.append(float(param_str))

print('Indices:', Ma_indices)
print('Ma list:', Ma_list)

# Optional: subsample for plotting (None = use all runs)
PLOT_SUBSET = None  # e.g. [0, 4, 8, ...] are positions into Ma_list

success_bools = []
final_depths = []
final_times = []
Ma_actual = []

if PLOT_SUBSET is not None:
    Ma_list = [Ma_list[j] for j in PLOT_SUBSET]
    Ma_indices = [Ma_indices[j] for j in PLOT_SUBSET]

for i in range(len(Ma_list)):
    try:
        print('Processing Ma=' + str(Ma_list[i]))
        filepath = snapshot_path_continuous(Ma_indices[i])
        with h5py.File(filepath, 'r') as f:
            s_data = f['tasks']['zb'][:]
            times = f['scales']['sim_time'][:]
        if s_data[-1] > 0:
            success_bools.append(True)
        else:
            success_bools.append(False)
        if success_bools[-1]:
            final_depths.append(0.0)
        else:
            final_depths.append(float(-s_data[-1, 0, 0, 0]))
        if success_bools[-1]:
            final_times.append(float(np.interp(0, s_data[:, 0, 0, 0], times)))
        else:
            final_times.append(np.inf)
        sd2 = np.squeeze(s_data)
        speeds = np.abs((sd2[1:-1] - sd2[0:-2]) / (times[1:-1] - times[0:-2]))
        Ma_actual.append(np.max(speeds))
        plt.figure(0)
        plt.plot(times, np.squeeze(s_data))
    except OSError:
        print(str(Ma_list[i]) + ' data not found')
        final_times.append(np.nan)
        success_bools.append(np.nan)
        final_depths.append(np.nan)

print('Stored Marangoni numbers:')
print(Ma_list)
print('Successfully reached interface?')
print(success_bools)
print('Depth at termination:')
print(final_depths)
print('Time at interface:')
print(final_times)

plt.xlim(0, 5.0)
plt.ylim(-1.0, 0)
plt.savefig('DC_trajectories.svg')

fig1 = plt.figure()
ax1 = plt.gca()
plt.scatter(Ma_list, final_times, marker='d')
plt.xlabel('Ma')
plt.ylabel('Transport time (T/T_diff)')
plt.xscale('log')
plt.yscale('log')
plt.savefig('DC_transport_times.svg')

